### Import Libraries

In [3]:
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
import mlflow.keras
import mlflow.keras
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam

### Configure MLflow Tracking URI and Experiment

In [4]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("AnomalyDetection")

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", mlflow.get_experiment_by_name("AnomalyDetection"))

Tracking URI: sqlite:///mlflow.db
Experiment: <Experiment: artifact_location='file:C:/Users/User/Desktop/usm/self learning/Detecting Anomaly/mlruns/1', creation_time=1779442328775, experiment_id='1', last_update_time=1779442328775, lifecycle_stage='active', name='AnomalyDetection', tags={}, trace_location=None, workspace='default'>


### Datasets

In [5]:
datasets = ["D1.csv", "D2.csv"]

### Models: Isolation Forest & Autoencoder

In [7]:
# Isolation Forest Function

def run_isolation_forest(train_X, test_X):
    model = IsolationForest(
        n_estimators=100,
        contamination="auto",
        random_state=42
    )
    model.fit(train_X)
    raw_preds = model.predict(test_X)
    
    # Convert:
    # 1 -> normal -> 0
    # -1 -> anomaly -> 1
    preds = np.where(raw_preds == -1, 1, 0)
    return model, preds

# Autoencoder Function
def run_autoencoder(train_X, train_y, test_X):
    # Train only on normal data
    normal_train = train_X[train_y == 0]
    input_dim = train_X.shape[1]
  
    # Encoder
    input_layer = Input(shape=(input_dim,))
    encoded = Dense(32, activation="relu")(input_layer)
    encoded = Dense(16, activation="relu")(encoded)
    encoded = Dense(8, activation="relu")(encoded)

    # Decoder
    decoded = Dense(16, activation="relu")(encoded)
    decoded = Dense(32, activation="relu")(decoded)
    decoded = Dense(input_dim, activation="sigmoid")(decoded)

    autoencoder = Model(inputs=input_layer, outputs=decoded)

    autoencoder.compile(
        optimizer=Adam(learning_rate=0.001),
        loss="mse"
    )

    autoencoder.fit(
        normal_train,
        normal_train,
        epochs=10,
        batch_size=256,
        shuffle=True,
        validation_split=0.1,
        verbose=1
    )

    # Reconstruction
    reconstructed = autoencoder.predict(test_X)

    # Reconstruction error
    mse = np.mean(np.power(test_X - reconstructed, 2), axis=1)

    # Threshold
    threshold = np.percentile(mse, 95)
    preds = (mse > threshold).astype(int)
    return autoencoder, preds

### Experiment with MLflow Tracking

In [11]:
for file in datasets:
    df = pd.read_csv(file)

    X = df.drop(["is_test", "target"], axis=1)
    y = df["target"]

    train_X = X[df["is_test"] == 0]
    test_X = X[df["is_test"] == 1]
    train_y = y[df["is_test"] == 0]
    test_y = y[df["is_test"] == 1]
    
    with mlflow.start_run(run_name=f"IsolationForest_{file}"):
        model, preds = run_isolation_forest(train_X, test_X)

        mlflow.log_param("dataset", file)
        mlflow.log_param("model", "Isolation Forest")

        mlflow.log_metric("accuracy", (preds == test_y).mean())
        mlflow.log_metric("precision", precision_score(test_y, preds))
        mlflow.log_metric("recall", recall_score(test_y, preds))
        mlflow.log_metric("f1", f1_score(test_y, preds))

        mlflow.sklearn.log_model(model, name="isolation_forest")

    with mlflow.start_run(run_name=f"Autoencoder_{file}"):
    
        model, preds = run_autoencoder(
            train_X,
            train_y,
            test_X
        )

        mlflow.log_param("dataset", file)
        mlflow.log_param("model", "Autoencoder")

        mlflow.log_metric("accuracy", accuracy_score(test_y, preds))
        mlflow.log_metric("precision", precision_score(test_y, preds))
        mlflow.log_metric("recall", recall_score(test_y, preds))
        mlflow.log_metric("f1_score", f1_score(test_y, preds))

        mlflow.keras.log_model(model, name="autoencoder")

2026/05/22 19:52:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Epoch 1/10
1368/1368 ━━━━━━━━━━━━━━━━━━━━ 29s 13ms/step - loss: 483042.1875 - val_loss: 460961.4688
Epoch 2/10
1368/1368 ━━━━━━━━━━━━━━━━━━━━ 17s 12ms/step - loss: 483042.1250 - val_loss: 460961.3125
Epoch 3/10
1368/1368 ━━━━━━━━━━━━━━━━━━━━ 21s 12ms/step - loss: 483042.3125 - val_loss: 460961.3125
Epoch 4/10
1368/1368 ━━━━━━━━━━━━━━━━━━━━ 18s 13ms/step - loss: 483041.9375 - val_loss: 460961.3125
Epoch 5/10
1368/1368 ━━━━━━━━━━━━━━━━━━━━ 18s 13ms/step - loss: 483041.9062 - val_loss: 460961.3125
Epoch 6/10
1368/1368 ━━━━━━━━━━━━━━━━━━━━ 20s 12ms/step - loss: 483042.0000 - val_loss: 460961.3125
Epoch 7/10
1368/1368 ━━━━━━━━━━━━━━━━━━━━ 19s 14ms/step - loss: 483041.9375 - val_loss: 460961.3125
Epoch 8/10
1368/1368 ━━━━━━━━━━━━━━━━━━━━ 19s 14ms/step - loss: 483042.0938 - val_loss: 460961.3125
Epoch 9/10
1368/1368 ━━━━━━━━━━━━━━━━━━━━ 18s 13ms/step - loss: 483042.3125 - val_loss: 460961.3125
Epoch 10/10
1368/1368 ━━━━━━━━━━━━━━━━━━━━ 17s 12ms/step - loss: 483042.2812 - val_loss: 460961.3125

2026/05/22 19:58:30 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.
2026/05/22 20:00:01 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Epoch 1/10
213/213 ━━━━━━━━━━━━━━━━━━━━ 14s 16ms/step - loss: 19889.4258 - val_loss: 20900.4824
Epoch 2/10
213/213 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - loss: 19888.5254 - val_loss: 20900.4160
Epoch 3/10
213/213 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - loss: 19888.5000 - val_loss: 20900.4082
Epoch 4/10
213/213 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - loss: 19888.5000 - val_loss: 20900.3965
Epoch 5/10
213/213 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - loss: 19888.4902 - val_loss: 20900.3926
Epoch 6/10
213/213 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - loss: 19888.4902 - val_loss: 20900.3926
Epoch 7/10
213/213 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - loss: 19888.4922 - val_loss: 20900.3906
Epoch 8/10
213/213 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - loss: 19888.4844 - val_loss: 20900.3906
Epoch 9/10
213/213 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - loss: 19888.4805 - val_loss: 20900.3906
Epoch 10/10
213/213 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - loss: 19888.4746 - val_loss: 20900.3887
2071/2071 ━━━━━━━━━━━━━━━━━━━━ 16s 8ms/step


2026/05/22 20:01:49 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.


### Launch MLflow Tracking UI

In [13]:
import subprocess

subprocess.Popen([
    "mlflow",
    "ui",
    "--backend-store-uri",
    "sqlite:///mlflow.db",
    "--port",
    "5001"
])

<Popen: returncode: None args: ['mlflow', 'ui', '--backend-store-uri', 'sqli...>